# Travel Reimbursement Approval Agent

A small, runnable prototype that evaluates employee travel reimbursement claims against the policy in Appendix A of the assignment and returns one structured decision per claim (`APPROVE`, `PARTIAL_APPROVE`, `REJECT`, `MANUAL_REVIEW`).

**How it works in one paragraph.** Every policy rule is encoded as a deterministic Python tool (limit checker, receipt checker, timeliness checker, approval-tier checker, policy lookup, output validator). An LLM (OpenAI via LangChain tool-calling) acts as the *orchestrator*: it reads the claim, decides which tools to call, combines their results, and submits a decision through a validator tool that enforces the output schema. Anything ambiguous or unvalidated falls back to `MANUAL_REVIEW`. Math and rule enforcement never live inside the LLM; the LLM only reasons about *how the rules combine* and writes the explanation.

---

## README

### Setup

Python 3.11+ and four packages. Nothing else is needed; policy and claims are embedded in the notebook.

```bash
pip install langchain langchain-openai python-dotenv ipykernel
```

### Required environment variables

Create a `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-5-mini     # optional, defaults to gpt-5-mini
```

### Run the demo

`Kernel → Restart & Run All`. No manual steps. The notebook:

1. Loads policy + the 5 sample claims (Appendix A / B, embedded below — no external files).
2. Runs the agent on each claim and records an audit trail (tools called, intermediate results).
3. Renders a dashboard under **## Dashboard**.
4. Prints the final JSON array in the **last code cell**.

### Key design choices

- **Deterministic tools, LLM orchestration.** Caps, eligibility, receipt rules and tier thresholds are code. The LLM decides *when* to call them and how to combine results, and writes the explanation.
- **Validator-as-tool.** The agent must call `submit_decision`; it rejects malformed output with an error message so the model can self-correct. If it still fails, the claim is routed to `MANUAL_REVIEW` by code.
- **Prefer Manual Review.** Business-class airfare, missing receipts, totals > $2,000, late submission, or any inconsistency route to `MANUAL_REVIEW`. Nothing is force-decided.
- **Zero extra dependencies.** Dashboard is plain HTML rendered with `IPython.display`.

See **Design Notes & Reasoning** at the end for trade-offs and next steps.

In [32]:
import os, json
from datetime import date
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from IPython.display import HTML, display

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

model = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"))

## 1. Policy context & claim intake

Policy (Appendix A) is stored as a dict keyed by stable rule id so the agent can retrieve and cite rules. Limits and thresholds are a separate small table. Claims (Appendix B) are plain JSON, which is the intake format; a CSV or API payload would be normalised into this same shape.

In [33]:
POLICY = {
    "POL-CAT-01": "Eligible categories (documented business purpose): airfare (economy only, see POL-AIR-01), lodging, meals (per-diem, see POL-PD-01), ground transport (taxi, rideshare, train, rental car, parking), conference/registration fees.",
    "POL-CAT-02": "Ineligible, never reimbursable, deducted in full: alcohol and minibar; spa, gym, personal entertainment; in-room movies, personal shopping, gifts; traffic fines, penalties, late fees; any personal (non-business) expense.",
    "POL-PD-01":  "Meals: maximum $75 per day. Amounts above the daily cap are deducted; the rest is reimbursed.",
    "POL-PD-02":  "Lodging: maximum $200 per night. Amounts above the nightly cap are deducted; the rest is reimbursed.",
    "POL-PD-03":  "Ground transport: maximum $50 per day. Amounts above the cap are deducted.",
    "POL-AIR-01": "Only economy class airfare is reimbursable. Business/first-class fares are a policy exception and must be routed to Manual Review (not auto-deducted, because a pre-approval may exist).",
    "POL-RCT-01": "Receipt required above $25: any single line item > $25 requires an attached itemized receipt. Airfare and lodging always require a receipt regardless of amount.",
    "POL-RCT-02": "Missing receipt handling: if a receipt is missing for an item that requires one, the item is not silently rejected; the claim is routed to Manual Review so the reviewer can request the receipt.",
    "POL-APR-01": "Auto-approve tier: total <= $500 may be auto-approved if fully compliant.",
    "POL-APR-02": "Manager tier: total > $500 and <= $2,000 is approvable when fully compliant.",
    "POL-APR-03": "Director / Manual-Review tier: total > $2,000 exceeds the agent's authority and must be routed to Manual Review even if otherwise compliant.",
    "POL-TIME-01": "Submission window: claims must be submitted within 30 days of the expense date. Late claims are routed to Manual Review.",
    "DECISION-GUIDANCE": "APPROVE: every item eligible, receipts present, within per-diem, total within POL-APR-01/02. PARTIAL_APPROVE: valid claim but some amounts exceed caps; reimburse up to cap, deduct excess. REJECT: items ineligible (POL-CAT-02) with nothing reimbursable. MANUAL_REVIEW: any ambiguity, policy exception, high value (POL-APR-03), missing required receipt (POL-RCT-02), or conflicting information. Prefer Manual Review over forcing a decision.",
}

LIMITS = {
    # category -> (cap per unit, unit, rule). unit is "day" (trip days inclusive) or "night" (trip days - 1)
    "meals":            (75.0,  "day",   "POL-PD-01"),
    "lodging":          (200.0, "night", "POL-PD-02"),
    "ground_transport": (50.0,  "day",   "POL-PD-03"),
}
ELIGIBLE   = {"airfare", "lodging", "meals", "ground_transport", "conference_fees"}
INELIGIBLE = {"alcohol", "minibar", "spa", "gym", "entertainment", "movies", "shopping", "gifts", "fines", "penalties", "late_fees", "personal"}
RECEIPT_THRESHOLD = 25.0
RECEIPT_ALWAYS    = {"airfare", "lodging"}
TIERS = [(500.0, "POL-APR-01", "auto"), (2000.0, "POL-APR-02", "manager"), (float("inf"), "POL-APR-03", "director_manual_review")]
SUBMISSION_WINDOW_DAYS = 30

CLAIMS = [
    {"claim_id": "CLM-001", "employee": "A. Rivera", "purpose": "Attend 2-day industry conference (business)",
     "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted": "2026-06-20",
     "items": [
        {"category": "airfare",         "description": "Round-trip economy airfare", "amount": 420.00, "receipt": True},
        {"category": "lodging",         "description": "Hotel, 2 nights @ $180",      "amount": 360.00, "receipt": True},
        {"category": "meals",           "description": "Meals, 3 days @ ~$60/day",    "amount": 180.00, "receipt": True},
        {"category": "conference_fees", "description": "Conference registration",     "amount": 150.00, "receipt": True}]},
    {"claim_id": "CLM-002", "employee": "B. Osei", "purpose": "Weekend hotel stay",
     "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted": "2026-06-25",
     "items": [
        {"category": "spa",     "description": "Hotel spa package", "amount": 300.00, "receipt": True},
        {"category": "minibar", "description": "In-room minibar",   "amount": 80.00,  "receipt": True}]},
    {"claim_id": "CLM-003", "employee": "C. Nakamura", "purpose": "Client site visit (business)",
     "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted": "2026-06-22",
     "items": [
        {"category": "airfare", "description": "Round-trip economy airfare", "amount": 300.00, "receipt": True},
        {"category": "lodging", "description": "Hotel, 2 nights @ $250",      "amount": 500.00, "receipt": True},
        {"category": "meals",   "description": "Meals, 2 days @ $70/day",     "amount": 140.00, "receipt": True}]},
    {"claim_id": "CLM-004", "employee": "D. Fischer", "purpose": "International vendor negotiation (business)",
     "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted": "2026-06-28",
     "items": [
        {"category": "airfare", "description": "Business-class international airfare", "amount": 2400.00, "receipt": True},
        {"category": "lodging", "description": "Hotel, 3 nights",                      "amount": 600.00,  "receipt": False}]},
    {"claim_id": "CLM-005", "employee": "E. Haddad", "purpose": "Client dinner / business development",
     "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted": "2026-06-24",
     "items": [
        {"category": "meals", "description": "Client dinner for 4 (business development)", "amount": 220.00, "receipt": False}]},
]
CLAIMS_BY_ID = {c["claim_id"]: c for c in CLAIMS}
print(f"{len(CLAIMS)} claims loaded, {len(POLICY) - 1} policy rules")

5 claims loaded, 12 policy rules


## 2. Tools

Six deterministic tools. Each returns JSON the LLM can reason over and cites the `POL-*` ids it applied. Two helpers handle date math (trip days are inclusive; nights = days − 1).

| Tool | Purpose |
|---|---|
| `lookup_policy` | Retrieve rule text by id (context grounding / RAG-lite) |
| `check_limits` | Per-line eligibility, per-diem caps, deductions, business-class flag |
| `check_receipts` | POL-RCT-01/02 receipt completeness |
| `check_timeliness` | POL-TIME-01 submission window |
| `check_approval_tier` | POL-APR-01/02/03 on the post-deduction total |
| `submit_decision` | Output validator; the only way the agent can finish |

In [34]:
def _d(s): return date.fromisoformat(s)
def _trip_days(c): return (_d(c["trip_end"]) - _d(c["trip_start"])).days + 1
def _nights(c): return max(_trip_days(c) - 1, 1)   # ponytail: 1-day trips can't have 0 nights of lodging cap; reviewer catches oddities

@tool
def lookup_policy(rule_ids: list[str]) -> dict:
    """Return the text of the given POL-* rule ids (or all rules if the list is empty). Use this to ground your decision in policy."""
    ids = rule_ids or list(POLICY)
    return {rid: POLICY.get(rid, "UNKNOWN RULE ID") for rid in ids}

@tool
def check_limits(claim_id: str) -> dict:
    """Apply category eligibility (POL-CAT-01/02), per-diem caps (POL-PD-*) and the airfare-class rule (POL-AIR-01) to every line item. Returns per-line reimbursable/deducted amounts, totals, policy refs and manual-review flags."""
    c = CLAIMS_BY_ID[claim_id]
    days, nights = _trip_days(c), _nights(c)
    lines, refs, flags = [], set(), []
    for it in c["items"]:
        cat, amt = it["category"], it["amount"]
        line = {"category": cat, "amount": amt, "reimbursable": 0.0, "deducted": 0.0, "note": ""}
        if cat in INELIGIBLE or cat not in ELIGIBLE:
            line.update(deducted=amt, note=f"ineligible category '{cat}' (POL-CAT-02)"); refs.add("POL-CAT-02")
        elif cat == "airfare" and any(k in it["description"].lower() for k in ("business", "first")):
            line.update(note="non-economy airfare: policy exception, requires Manual Review (POL-AIR-01)")
            flags.append("POL-AIR-01 non-economy airfare"); refs.add("POL-AIR-01")
        elif cat in LIMITS:
            cap, unit, rule = LIMITS[cat]
            allowed = cap * (days if unit == "day" else nights)
            reimb = min(amt, allowed)
            line.update(reimbursable=reimb, deducted=round(amt - reimb, 2),
                        note=f"cap ${cap:.0f}/{unit} x {days if unit=='day' else nights} = ${allowed:.2f} ({rule})")
            refs.update({"POL-CAT-01", rule})
        else:
            line.update(reimbursable=amt, note="eligible (POL-CAT-01)"); refs.add("POL-CAT-01")
        lines.append(line)
    return {"trip_days": days, "nights": nights, "lines": lines,
            "total_claimed": round(sum(l["amount"] for l in lines), 2),
            "total_reimbursable": round(sum(l["reimbursable"] for l in lines), 2),
            "total_deducted": round(sum(l["deducted"] for l in lines), 2),
            "manual_review_flags": flags, "policy_refs": sorted(refs)}

@tool
def check_receipts(claim_id: str) -> dict:
    """Check receipt completeness (POL-RCT-01: receipt required > $25, always for airfare/lodging). Missing receipts mean Manual Review (POL-RCT-02), not rejection."""
    c = CLAIMS_BY_ID[claim_id]
    missing = [f"{it['category']}: {it['description']} (${it['amount']:.2f})" for it in c["items"]
               if not it["receipt"] and (it["amount"] > RECEIPT_THRESHOLD or it["category"] in RECEIPT_ALWAYS)]
    return {"missing_docs": missing, "requires_manual_review": bool(missing),
            "policy_refs": ["POL-RCT-01"] + (["POL-RCT-02"] if missing else [])}

@tool
def check_timeliness(claim_id: str) -> dict:
    """Check POL-TIME-01: claim must be submitted within 30 days of the expense (trip end) date."""
    c = CLAIMS_BY_ID[claim_id]
    lag = (_d(c["submitted"]) - _d(c["trip_end"])).days
    return {"days_after_trip": lag, "on_time": 0 <= lag <= SUBMISSION_WINDOW_DAYS, "policy_refs": ["POL-TIME-01"]}

@tool
def check_approval_tier(total_reimbursable: float) -> dict:
    """Classify the post-deduction reimbursable total into an approval tier (POL-APR-01/02/03). Totals > $2000 require Manual Review."""
    for limit, rule, tier in TIERS:
        if total_reimbursable <= limit:
            return {"tier": tier, "policy_ref": rule, "requires_manual_review": tier == "director_manual_review"}

class Decision(BaseModel):
    claim_id: str
    decision: Literal["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
    approved_amount: float = Field(ge=0)
    deducted_amount: float = Field(ge=0)
    missing_docs: list[str]
    policy_refs: list[str]
    confidence: float = Field(ge=0, le=1)
    explanation: str
    tools_used: list[str]

@tool(args_schema=Decision)
def submit_decision(**kwargs) -> dict:
    """Submit the final decision for a claim. Validates the schema and business invariants. Returns {"ok": true} or {"ok": false, "errors": [...]}; fix the errors and resubmit."""
    d = Decision(**kwargs)
    errors = []
    c = CLAIMS_BY_ID.get(d.claim_id)
    if not c: errors.append(f"unknown claim_id {d.claim_id}")
    else:
        total = round(sum(i["amount"] for i in c["items"]), 2)
        # check: can round in below line can introduce bug.
        if d.decision in ("APPROVE", "PARTIAL_APPROVE", "REJECT") and round(d.approved_amount + d.deducted_amount, 2) != total:
            errors.append(f"approved + deducted must equal total claimed {total}")
        if d.decision == "APPROVE" and d.deducted_amount != 0: errors.append("APPROVE must have deducted_amount 0")
        if d.decision == "REJECT" and d.approved_amount != 0: errors.append("REJECT must have approved_amount 0")
        if d.decision == "MANUAL_REVIEW" and d.approved_amount != 0: errors.append("MANUAL_REVIEW must have approved_amount 0 (nothing is paid until a human decides)")
    unknown = [r for r in d.policy_refs if r not in POLICY]
    if unknown: errors.append(f"unknown policy refs {unknown}")
    return {"ok": not errors, "errors": errors}

TOOLS = [lookup_policy, check_limits, check_receipts, check_timeliness, check_approval_tier, submit_decision]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

## 3. Agent

A plain tool-calling loop (no framework graph needed for a linear-ish task). The system prompt tells the model what each tool is for and how the rules combine; the model decides the order and which tools are relevant. The loop ends only when `submit_decision` returns `ok: true`. If the model exhausts its step budget or never produces a valid decision, code falls back to `MANUAL_REVIEW` with low confidence, so the pipeline never emits an unvalidated result.

In [35]:
SYSTEM_PROMPT = f"""You are a travel reimbursement approval agent. Evaluate ONE claim against company policy using the tools provided.

Rules of engagement:
- Never compute caps, eligibility or totals yourself; always call check_limits, check_receipts, check_timeliness, then check_approval_tier on the reimbursable total from check_limits.
- Call lookup_policy when you need the exact wording of a rule you intend to cite.
- Decision guidance: {POLICY['DECISION-GUIDANCE']}
- Any manual_review flag, missing receipt, late submission, director tier, or conflicting/ambiguous information => MANUAL_REVIEW. Prefer MANUAL_REVIEW over forcing a decision.
- approved_amount for MANUAL_REVIEW is 0 (nothing is paid until a reviewer decides); state in the explanation what the amount would be if approved.
- For APPROVE/PARTIAL_APPROVE/REJECT: approved_amount + deducted_amount must equal the total claimed.
- confidence: 0-1, how certain you are the decision is right and complete; lower it when information is missing or ambiguous.
- tools_used: list the names of the tools you actually called (excluding submit_decision).
- Finish by calling submit_decision. If it returns errors, fix them and call it again. Do not reply with free text.
"""

def evaluate_claim(claim: dict, max_steps: int = 10) -> tuple[dict, list]:
    """Returns (decision_dict, audit_trail)."""
    llm = model.bind_tools(TOOLS)
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(f"Evaluate this claim:\n{json.dumps(claim, indent=2)}")]
    trail, tools_called = [], []
    for _ in range(max_steps):
        ai = llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            trail.append({"step": "assistant_text", "content": ai.content}); break
        for tc in ai.tool_calls:
            result = TOOLS_BY_NAME[tc["name"]].invoke(tc["args"])
            trail.append({"tool": tc["name"], "args": tc["args"], "result": result})
            messages.append(ToolMessage(content=json.dumps(result), tool_call_id=tc["id"]))
            if tc["name"] == "submit_decision" and result["ok"]:
                d = Decision(**tc["args"]).model_dump()
                d["tools_used"] = sorted(set(tools_called))  # trust the audit trail, not the model's self-report
                return d, trail
            if tc["name"] != "submit_decision":
                tools_called.append(tc["name"])
    # Fallback: no validated decision -> never force one
    total = round(sum(i["amount"] for i in claim["items"]), 2)
    return {"claim_id": claim["claim_id"], "decision": "MANUAL_REVIEW", "approved_amount": 0.0, "deducted_amount": 0.0,
            "missing_docs": [], "policy_refs": ["DECISION-GUIDANCE"], "confidence": 0.2,
            "explanation": f"Agent did not produce a validated decision within {max_steps} steps; routed to Manual Review (total claimed ${total:.2f}).",
            "tools_used": sorted(set(tools_called))}, trail

In [37]:
RESULTS, AUDIT = [], {}
for claim in CLAIMS:
    decision, trail = evaluate_claim(claim)
    RESULTS.append(decision); AUDIT[claim["claim_id"]] = trail
    print(f"{decision['claim_id']}: {decision['decision']:15s} approved=${decision['approved_amount']:>8.2f} deducted=${decision['deducted_amount']:>7.2f} conf={decision['confidence']:.2f}")

CLM-001: APPROVE         approved=$ 1110.00 deducted=$   0.00 conf=0.95
CLM-002: REJECT          approved=$    0.00 deducted=$ 380.00 conf=0.95
CLM-003: PARTIAL_APPROVE approved=$  840.00 deducted=$ 100.00 conf=0.90
CLM-004: MANUAL_REVIEW   approved=$    0.00 deducted=$   0.00 conf=0.65
CLM-005: MANUAL_REVIEW   approved=$    0.00 deducted=$ 220.00 conf=0.60


## 4. Sample outputs & audit trail

For each claim: the tools the agent chose to call (in order), the intermediate checks, and the generated explanation. This is the evidence that the LLM is orchestrating tools rather than guessing.

In [38]:
for r in RESULTS:
    print("=" * 100)
    print(f"{r['claim_id']}  ->  {r['decision']}  (confidence {r['confidence']:.2f})")
    print(f"  approved ${r['approved_amount']:.2f} | deducted ${r['deducted_amount']:.2f} | missing_docs={r['missing_docs']} | refs={r['policy_refs']}")
    print(f"  explanation: {r['explanation']}")
    print("  tool sequence:")
    for step in AUDIT[r["claim_id"]]:
        if "tool" in step:
            summary = {k: v for k, v in step["result"].items() if k in ("total_reimbursable", "total_deducted", "manual_review_flags", "missing_docs", "on_time", "tier", "ok", "errors")}
            print(f"    - {step['tool']}({step['args']}) -> {summary}")

CLM-001  ->  APPROVE  (confidence 0.95)
  approved $1110.00 | deducted $0.00 | missing_docs=[] | refs=['POL-CAT-01', 'POL-PD-01', 'POL-PD-02', 'POL-RCT-01', 'POL-TIME-01', 'POL-APR-02']
  explanation: All line items are eligible under POL-CAT-01 (airfare, lodging, meals, conference fees). Per-diem checks show meals ($180) are within the $75/day cap (max $225 for 3 days) per POL-PD-01, and lodging ($360) is within the $200/night cap (max $400 for 2 nights) per POL-PD-02. Airfare included and is economy with receipt; receipts are present for all items (POL-RCT-01). Claim was submitted 8 days after trip end, within the 30-day window (POL-TIME-01). Total reimbursable $1,110.00 is within the manager approval tier (POL-APR-02). No deductions required. Approve full amount of $1,110.00.
  tool sequence:
    - check_limits({'claim_id': 'CLM-001'}) -> {'total_reimbursable': 1110.0, 'total_deducted': 0.0, 'manual_review_flags': []}
    - check_receipts({'claim_id': 'CLM-001'}) -> {'missing_docs':

## Dashboard

Minimal, data-driven summary derived from `RESULTS`: decision breakdown, approved vs deducted vs pending-review amounts, and a per-claim table. Rendered as plain HTML (no plotting dependency).

In [39]:
def dashboard(results):
    counts = {d: sum(r["decision"] == d for r in results) for d in ("APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW")}
    claimed = {r["claim_id"]: sum(i["amount"] for i in CLAIMS_BY_ID[r["claim_id"]]["items"]) for r in results}
    approved = sum(r["approved_amount"] for r in results)
    deducted = sum(r["deducted_amount"] for r in results)
    pending  = sum(claimed[r["claim_id"]] for r in results if r["decision"] == "MANUAL_REVIEW")
    total    = sum(claimed.values())
    color = {"APPROVE": "#2e8b57", "PARTIAL_APPROVE": "#e0a800", "REJECT": "#c0392b", "MANUAL_REVIEW": "#5b7db1"}
    bar = lambda v, mx, c: f'<div style="background:{c};height:14px;width:{(v/mx*100) if mx else 0:.0f}%;display:inline-block"></div> {v}'
    rows = "".join(
        f"<tr><td>{r['claim_id']}</td><td style='color:{color[r['decision']]};font-weight:bold'>{r['decision']}</td>"
        f"<td>${claimed[r['claim_id']]:,.2f}</td><td>${r['approved_amount']:,.2f}</td><td>${r['deducted_amount']:,.2f}</td>"
        f"<td>{r['confidence']:.2f}</td><td>{', '.join(r['policy_refs'])}</td><td>{len(r['tools_used'])}</td></tr>" for r in results)
    html = f"""
    <style>.rb table{{border-collapse:collapse;font-family:monospace}} .rb td,.rb th{{border:1px solid #ccc;padding:4px 8px;text-align:left}}</style>
    <div class="rb">
    <h3>Decision breakdown ({len(results)} claims)</h3>
    <table>{''.join(f"<tr><td>{d}</td><td style='width:260px'>{bar(n, max(counts.values()), color[d])}</td></tr>" for d, n in counts.items())}</table>
    <h3>Amounts (USD)</h3>
    <table>
      <tr><td>Total claimed</td><td>${total:,.2f}</td><td style='width:260px'></td></tr>
      <tr><td>Approved</td><td>${approved:,.2f}</td><td>{bar(round(approved), round(total), color['APPROVE'])}</td></tr>
      <tr><td>Deducted</td><td>${deducted:,.2f}</td><td>{bar(round(deducted), round(total), color['REJECT'])}</td></tr>
      <tr><td>Pending manual review</td><td>${pending:,.2f}</td><td>{bar(round(pending), round(total), color['MANUAL_REVIEW'])}</td></tr>
    </table>
    <h3>Per claim</h3>
    <table><tr><th>Claim</th><th>Decision</th><th>Claimed</th><th>Approved</th><th>Deducted</th><th>Confidence</th><th>Policy refs</th><th>Tools</th></tr>{rows}</table>
    </div>"""
    return HTML(html)

display(dashboard(RESULTS))

## Design Notes & Reasoning

### Architecture: deterministic tools, LLM as orchestrator
- Every rule in Appendix A maps to code in a tool. Dollar math, caps and eligibility are never left to the model; a wrong number in a reimbursement system is worse than a slow one.
- The LLM's job is *workflow control*: decide which tools to call, read the results, resolve how rules combine (e.g. "deductions exist **and** a receipt is missing" → Manual Review beats Partial Approve), and write a human-readable explanation with citations.
- `submit_decision` is both the output schema and a business-invariant validator (`approved + deducted == claimed`, `REJECT ⇒ approved 0`, known `POL-*` refs). The model must pass validation to finish, which turns schema errors into a self-correction loop instead of a crash.
- `tools_used` is taken from the audit trail, not the model's self-report, so it cannot hallucinate tool usage.

### Why these cases go to Manual Review
| Trigger | Rule | Reasoning |
|---|---|---|
| Non-economy airfare (CLM-004) | POL-AIR-01 | A pre-approval may exist; auto-deducting $2,400 would be wrong if it does. |
| Missing required receipt (CLM-004, CLM-005) | POL-RCT-02 | Policy says request the receipt, not reject. |
| Reimbursable total > $2,000 | POL-APR-03 | Outside the agent's authority regardless of compliance. |
| Submission > 30 days after expense | POL-TIME-01 | Policy routes late claims to a human. |
| Ambiguity (CLM-005: "dinner for 4", one-day trip, $220 vs $75 cap) | Decision Guidance | Could be 4 × per-diem or a client-entertainment exception; the policy does not say. Forcing a $145 deduction would be a guess. |
| Agent fails to produce a validated decision | fallback in code | Never emit an unvalidated result. |

Expected outcomes for the five sample claims (used as the sanity check when reviewing the run above):
`CLM-001 APPROVE $1,110` · `CLM-002 REJECT $0 (deduct $380)` · `CLM-003 PARTIAL_APPROVE $840 (deduct $100 lodging)` · `CLM-004 MANUAL_REVIEW` · `CLM-005 MANUAL_REVIEW`.

### Assumptions & simplifications
- Trip days are inclusive of start and end (3 days for 06-10 → 06-12); nights = days − 1 (minimum 1). Per-diem caps are applied to the category total over the trip, not per individual day, because claims are given as category totals.
- "Expense date" for POL-TIME-01 is the trip end date.
- Airfare class is detected from the description text ("business"/"first"); a real system would have a structured `fare_class` field or we need to use llm here as well.
- `approved_amount` is 0 for `MANUAL_REVIEW` because nothing is paid until a human decides; the explanation states the would-be amount.
- Category eligibility is an exact-match lookup on a fixed vocabulary; unknown categories are treated as ineligible and surface as deductions the reviewer can see.
- Policy retrieval is a dict lookup, not a vector store. Twelve rules with stable ids do not need embeddings; adding them would be over-engineering.
- No persistence, no auth, no concurrency, no real receipts (attachment is a boolean flag).

### Known gaps
- Duplicate-claim detection is not implemented (no cross-claim history in the sample set).
- No OCR / receipt-content verification; "receipt attached" is trusted.
- Model non-determinism: the deterministic tools make the *amounts* stable, but explanations and confidence vary between runs.
- No multi-currency handling.

### What I would improve next
1. Add an evaluation cell that asserts the five expected decisions above and runs on every change (basic regression test).
2. Structured `fare_class`, per-day meal lines and a `duplicate_detector` tool backed by a claims history table.
3. A reviewer queue UI: Manual Review items with the agent's pre-computed would-be amount and reason codes, so the human only has to confirm.
4. Confidence calibration against a labelled set, and reason codes (`MISSING_RECEIPT`, `OVER_AUTHORITY`, …) as a first-class field.

## Final structured results

One object per provided claim with exactly the required fields: `claim_id, decision, approved_amount, deducted_amount, missing_docs, policy_refs, confidence, explanation, tools_used`.

In [ ]:
FIELDS = ["claim_id", "decision", "approved_amount", "deducted_amount", "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"]
FINAL = [{k: r[k] for k in FIELDS} for r in RESULTS]
assert len(FINAL) == len(CLAIMS) and all(set(r) == set(FIELDS) for r in FINAL)
print(json.dumps(FINAL, indent=2))

[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-CAT-01",
      "POL-PD-01",
      "POL-PD-02",
      "POL-RCT-01",
      "POL-TIME-01",
      "POL-APR-02"
    ],
    "confidence": 0.95,
    "explanation": "All line items are eligible under POL-CAT-01 (airfare, lodging, meals, conference fees). Per-diem checks show meals ($180) are within the $75/day cap (max $225 for 3 days) per POL-PD-01, and lodging ($360) is within the $200/night cap (max $400 for 2 nights) per POL-PD-02. Airfare included and is economy with receipt; receipts are present for all items (POL-RCT-01). Claim was submitted 8 days after trip end, within the 30-day window (POL-TIME-01). Total reimbursable $1,110.00 is within the manager approval tier (POL-APR-02). No deductions required. Approve full amount of $1,110.00.",
    "tools_used": [
      "check_approval_tier",
      "check_limits",
     